# 99.7g User Story 908: SMA Kinetics Transferred from Another Simulator

## Contents

**Tools:** CADET-Python; reproduced forum-case model; integrator metrics; NumPy; Matplotlib; saved HDF5 simulation results.

**Python used:** Imports, paths, function definitions, `**kwargs`, exception handling, dictionaries, loops and comprehensions, array slicing, numerical operations and plotting.

**Introduces:** SMA kinetic conventions, reading `IDA_ERR_FAIL`, diagnosis at a section transition, `CONSISTENT_INIT_MODE`, catastrophic cancellation in a binding equation, and the rapid-equilibrium reformulation.

1. The failure: what the error message already tells us
2. Two hypotheses
3. The fix — part 1: what could make it run? · part 2: what does make it run? · part 3: why?
4. The actual answer: `IS_KINETIC = False`

An algebraically correct parameter translation can produce a numerically unusable model. We reproduce the failure, change one setting at a time, look at the numbers the binding equation is built from, and end up back at the model rather than at the solver.

## Fail case: a kinetic constant carried over from another simulator

*Forum thread: [Help with k_kin in SMA isotherm](https://forum.cadet-web.de/t/help-with-k-kin-in-sma-isotherm/908)*

A user wanted to reproduce a CEX load-wash-elute run from GoSilico in CADET.
GoSilico reported four SMA numbers:

| | |
|---|---|
| Equilibrium (SMA) | $k_{eq} = 7 \cdot 10^{-7}$ |
| Kinetic (SMA) | $k_{kin} = 1.8 \cdot 10^{-8}$ |
| Charge (SMA) | $\nu = 10.11$ |
| Shielding (SMA) | $\sigma = 0$ |

GoSilico writes the kinetic SMA with the rate constant on the left,

$$ k_{kin}\,\frac{\mathrm{d}q_i}{\mathrm{d}t} \;=\; k_{eq,i}\,\bar{q}^{\,\nu_i} c_{p,i} \;-\; c_{salt}^{\,\nu_i} q_i , $$

whereas CADET puts the two rate constants on the right,

$$ \frac{\mathrm{d}q_i}{\mathrm{d}t} \;=\; k_{a,i}\,\bar{q}^{\,\nu_i} c_{p,i} \;-\; k_{d,i}\,c_{salt}^{\,\nu_i} q_i . $$

Dividing the first equation by $k_{kin}$ gives the obvious translation

$$ k_a = \frac{k_{eq}}{k_{kin}} = 38.9, \qquad k_d = \frac{1}{k_{kin}} = 5.6 \cdot 10^{7} . $$

That translation is algebraically correct. **The simulation fails anyway.**

In [ ]:
# Change directory to cwd and add to system path
import os
import sys

root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')
os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from cadet import Cadet

import utility.cadet_runtime as cadet_runtime
from utility import setting_forum908_SMA_kkin as case
from utility import stiffness_metrics as sm

# The notebooks of this chapter need CADET-Core 6.0 or newer. configure()
# takes CADET_PATH if it is set and the current environment otherwise.
cadet_runtime.configure()

WORK = Path("generated") / "user_stories" / "fail_case_908_sma_kkin"
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
def simulate(label, quiet=False, **kwargs):
    """Run one variant of the case and return its integrator metrics.

    Returns None if the simulation failed. Unless `quiet` is set, CADET's
    stderr and its log are both printed -- the warnings are only in the log.
    """
    simulation = Cadet()
    simulation.root = case.get_model(**kwargs)
    simulation.filename = str(WORK / (label.replace(" ", "_") + ".h5"))

    # What run_and_measure() does, unrolled, so that the ReturnInformation
    # survives: it carries error_message (stderr) and log (stdout).
    sm.enable_idas_meta(simulation)
    simulation.save()
    info = simulation.run_simulation()

    if not quiet:
        if info.error_message.strip():
            print(info.error_message.strip().splitlines()[0])
        # CADET repeats the same warning once per failed attempt.
        for line in dict.fromkeys(info.log.strip().splitlines()):
            print("   ", line)

    if info.return_code != 0:
        return None

    return sm.measure(simulation.filename, label=label)


def solution_of(label):
    """Read back the solution group of a run that has been simulated."""
    loaded = Cadet()
    loaded.filename = str(WORK / (label.replace(" ", "_") + ".h5"))
    loaded.load_from_file()
    return loaded.root.output.solution

In [ ]:
baseline = simulate("baseline", write_solution_bulk=1,
                    write_solution_particle=1, write_solution_solid=1)

solution = solution_of("baseline")
times = solution.solution_times
column = solution.unit_000

figure, outlet_axis = plt.subplots(
    figsize=(6.5, 4.2), constrained_layout=True)

outlet_axis.plot(times / 60, column.solution_outlet[:, 1],
                 color=sm.COLORS[0], lw=2)
outlet_axis.axvline(case.WASH_TIME / 60, color=sm.SECTION_COLOR,
                    lw=0.8, ls="-.")

outlet_axis.set_xlabel("time / min")
outlet_axis.set_ylabel(r"protein at the outlet / mol m$^{-3}$")
outlet_axis.set_title(f"the run stops at t = {times[-1]:.1f} s")
outlet_axis.grid(True, color=sm.GRID_COLOR, lw=0.5)

plt.show()

## 1. The failure

`IDA_ERR_FAIL` at $t = 713.22$ s, and the step size at that moment is
$h = 6 \cdot 10^{-14}$ s. Three things are worth reading out of the log before
touching anything:

- **713.22 s is the first section transition**, where the protein feed is
  switched off and the wash begins.
- **`h` can no longer be reduced.** At $t = 713.22$ s the spacing between
  neighbouring double precision numbers is $1.1 \cdot 10^{-13}$ s, so a step of
  $6 \cdot 10^{-14}$ s is already at the resolution of the clock. The
  integrator stopped because it could not pass its local error test at any
  step size it can represent.
- The log also reports **`Residual has NAN or infinite value in unit 0`**.

## 2. The hypothesis

- **A)** NAN or infinite values are a strong indication that negative values
  occurred during the simulation, from which an undefined power was taken, for
  example in the binding or reaction model.
- **B)** A failure at a section transition is a strong indicator that the
  consistent initialization went wrong.

## 3. The fix — part 1: what could make it run?

- The SMA binding model takes a power to the characteristic charge
  $\nu = 10.11$. To test hypothesis A we can **change $\nu$ to an integer**,
  e.g. 10.0, for which a negative base is still well defined.
- Everything else gets **one change at a time**: the spatial grid, the
  tolerances, the first step, the two limits that govern how hard the
  integrator is allowed to try, and the feed itself.
- Hypothesis B gets its own experiment in the next cell but one.

In [ ]:
variants = {
    "baseline": {},
    "nu = 10 (integer)": dict(nu=10.0),
    "ncol = 20": dict(ncol=20),
    "ncol = 200": dict(ncol=200),
    "abstol = 1e-6": dict(abstol=1e-6),
    "abstol = 1e-10": dict(abstol=1e-10),
    "init_step_size = 1e-15": dict(init_step_size=1e-15),
    "max_errtest_fail = 10": dict(max_errtest_fail=10),
    "max_errtest_fail = 15": dict(max_errtest_fail=15),
    "max_newton_iter = 1000": dict(max_newton_iter=1000),
    "no protein in feed": dict(c_protein=0.0),
}

results = {name: simulate(name, quiet=True, **kwargs)
           for name, kwargs in variants.items()}

print(f"{'variant':<24} {'result':>8} {'steps':>7} {'t_cpu [s]':>10}")
print("-" * 52)
for name, metrics in results.items():
    if metrics is None:
        print(f"{name:<24} {'failed':>8} {'':>7} {'':>10}")
    else:
        print(f"{name:<24} {'ok':>8} {metrics.n_steps:>7d} "
              f"{metrics.compute_time:>10.2f}")

Now the consistent initialization. `CONSISTENT_INIT_MODE` selects what CADET
does at $t = 0$ and at every section transition: nothing, a *full* solve of the
algebraic subsystem followed by all initial time derivatives, or a *lean* one,
which — in CADET's own words — "updates only a part of the time derivative
vector", so the result "is not guaranteed to be consistent". Modes 2 to 7
combine a choice for $t = 0$ with a different one for the transitions.

In [ ]:
init_modes = {
    0: "none",
    1: "full (default)",
    2: "full once, none afterwards",
    3: "lean",
    4: "lean once, none afterwards",
    5: "full once, lean afterwards",
    6: "none once, full afterwards",
    7: "none once, lean afterwards",
}

print(f"{'mode':>4}  {'meaning':<28} {'result':>8}")
print("-" * 44)
for mode, description in init_modes.items():
    metrics = simulate(f"init_mode_{mode}", quiet=True,
                       consistent_init_mode=mode)
    print(f"{mode:>4}  {description:<28} {'ok' if metrics else 'failed':>8}")

## 3. The fix — part 2: what does make it run?

- **The spatial discretization is irrelevant:** `ncol` 20 and 200 both fail, so
  this is not a resolution problem.
- **Hypothesis A is out.** With $\nu = 10$ the run still fails — and the NAN
  warning disappears from the log. Negative bases do occur, but they are a
  *symptom* of the integrator flailing, not the reason it flails.
- **Loosening `abstol` helps, tightening it hurts.** $10^{-6}$ runs,
  $10^{-7}$, the default $10^{-8}$ and $10^{-10}$ all fail. Something in the
  equations cannot be evaluated accurately enough to satisfy the tighter
  error test.
- **`init_step_size = 1e-15` works.** Entering the new section on a step nine
  orders of magnitude smaller than the default gets past the transition.
- **`max_newton_iter` does nothing**, even at 1000. More Newton iterations help
  when the nonlinear solve converges slowly; here it is not the nonlinear
  solve that fails.
- **`max_errtest_fail` does, but only above its own default.** The CADET
  default is already 10 (inherited from IDAS), so the `= 10` row changes
  nothing — it is in the table as a reminder to check defaults before reading
  a result. From 15 upwards the run completes: IDAS is allowed to fail its
  local error test a few more times, halving $h$ each time, and eventually
  crawls through the transition.
- **Without protein in the feed there is no problem at all**, which puts the
  trouble in the protein binding equation rather than in the salt or the
  transport.
- **Hypothesis B is confirmed, and sharpened.** Exactly two of the eight modes
  work, 3 and 5, and they are exactly the two that apply *lean* initialization
  **at the transition**. Full fails there (modes 1 and 6) and so does none
  (modes 0, 2 and 4). At $t = 0$, on the other hand, none is not enough
  (modes 6 and 7 fail where 3 and 5 succeed).

Everything that helps buys the same thing: it avoids demanding an accurate
answer from the full initialization at that one moment. So the question is
what that computation actually evaluates.

## 3. The fix — part 3: why does it run now?

The first section ran fine. **Let us look at the values** that go into the
**binding equation** at the moment the full initialization has to solve it.
The following code prints the pore liquid and solid phases just before the
section transition, together with the adsorption and desorption terms built
from them.

In [ ]:
run = Cadet()
run.root = case.get_model(consistent_init_mode=3,
                          write_solution_particle=1, write_solution_solid=1,
                          n_solution_times=4001)
run.filename = str(WORK / "state.h5")
run.save()
run.run_simulation()
run.load_from_file()

solution = run.root.output.solution.unit_000
times = run.root.output.solution.solution_times

# the last output time before the section transition
index = int(np.argmin(np.abs(times - case.WASH_TIME)))
c_particle = solution.solution_particle[index]
q_solid = solution.solution_solid[index]

k_a = case.K_EQ / case.K_KIN
k_d = 1.0 / case.K_KIN

c_salt, c_protein = c_particle[:, 0], c_particle[:, 1]
q_protein = q_solid[:, 1]
q_bar = case.CAPACITY - case.NU * q_protein

adsorption = k_a * c_protein * q_bar ** case.NU
desorption = k_d * q_protein * c_salt ** case.NU
net = adsorption - desorption

largest = np.maximum(np.abs(adsorption), np.abs(desorption))
print(f"state at t = {times[index]:.2f} s, section transition at {case.WASH_TIME} s")
print(f"k_a = {k_a:.4g},  k_d = {k_d:.4g},  nu = {case.NU}\n")
print(f"{'cell':>5} {'c_p protein':>13} {'q protein':>11} "
      f"{'adsorption':>13} {'desorption':>13} {'difference':>13} {'rel.':>9}")
print("-" * 82)
for cell in [0, 1, 25, 50, 75, 99]:
    print(f"{cell:>5} {c_protein[cell]:>13.3e} {q_protein[cell]:>11.3e} "
          f"{adsorption[cell]:>13.5e} {desorption[cell]:>13.5e} "
          f"{net[cell]:>13.3e} {abs(net[cell]) / largest[cell]:>9.1e}")

print(f"\nlargest term in the column          : {largest.max():.3e}")
print(f"machine epsilon times that term     : {np.finfo(float).eps * largest.max():.3e}")
print(f"   -> that is the absolute noise floor of 'adsorption - desorption'")
print(f"smallest relative difference        : "
      f"{(np.abs(net) / largest).min():.1e}   (eps = {np.finfo(float).eps:.1e})")
print(f"physical rate d q/d t at that cell  : "
      f"{np.gradient(solution.solution_solid[:, 0, 1], times)[index]:.3e}")

In the first cell of the column, adsorption and desorption are both
$7.0 \cdot 10^{17}$ and they agree to about **fifteen decimal digits**. The
absolute noise floor of that subtraction — machine epsilon times the terms
themselves — is of order $10^{2}$.

The quantity being computed is $\mathrm{d}q/\mathrm{d}t$, and its physical
value right there is $1.2 \cdot 10^{-3}$ mol m$^{-3}$ s$^{-1}$. **The binding
rate is five orders of magnitude smaller than the rounding error of the
expression that produces it.** This is *catastrophic cancellation*: not a bug
and not a bad grid, but a property of these equations at this state, and hence
a precision problem that no solver setting can remove.

That also explains part 2 in one line. The full consistent initialization
solves for time derivatives using exactly this expression and then has to meet
a tolerance with the result; lean initialization does not recompute all of
them, a looser `abstol` stops asking for accuracy that is not there, and a
tiny first step lets the first step after the restart pass its error test.
All of them work around the same arithmetic rather than fixing it.

## 4. The actual answer: `IS_KINETIC = False`

If the adsorption and desorption rates are so large that $k_{kin}$ is not a physical parameter of this system at all.
Instead, the rapid-equilibrium assumption applies.

In rapid equilibrium only the ratio $k_a / k_d$ enters, so the translation
problem disappears: set $k_a = k_{eq}$ and $k_d = 1$.
<!-- #endregion -->

In [ ]:
kinetic = simulate("kinetic_lean", consistent_init_mode=3)
equilibrium = simulate("equilibrium", is_kinetic=False,
                       sma_ka=case.K_EQ, sma_kd=1.0)

times = solution_of("equilibrium").solution_times
outlets = {label: solution_of(label).unit_000.solution_outlet
           for label in ["kinetic_lean", "equilibrium"]}

difference = np.linalg.norm(
    outlets["kinetic_lean"][:, 1] - outlets["equilibrium"][:, 1]
) / np.linalg.norm(outlets["equilibrium"][:, 1])

print(f"kinetic (lean init)  : {kinetic.n_steps} steps, "
      f"{kinetic.compute_time:.2f} s")
print(f"rapid equilibrium    : {equilibrium.n_steps} steps, "
      f"{equilibrium.compute_time:.2f} s")
print(f"relative difference between the two chromatograms: {difference:.2e}")

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.5))
salt_axis = axis.twinx()

salt_axis.plot(times / 60, outlets["equilibrium"][:, 0], ":",
               color="grey", lw=1.2, label="salt")
axis.plot(times / 60, outlets["equilibrium"][:, 1],
          color=sm.COLORS[0], lw=2, label=r"rapid equilibrium")
axis.plot(times / 60, outlets["kinetic_lean"][:, 1], "--",
          color=sm.COLORS[1], lw=1.5,
          label=r"kinetic, $k_{kin} = 1.8 \cdot 10^{-8}$")

for boundary in [case.WASH_TIME, case.ELUTION_TIME]:
    axis.axvline(boundary / 60, color=sm.SECTION_COLOR, lw=0.8, ls="-.")

axis.set_xlabel("time / min")
axis.set_ylabel(r"protein / mol m$^{-3}$")
salt_axis.set_ylabel(r"salt / mol m$^{-3}$")
axis.set_title("the two models are the same model")
# One legend for both axes, so the salt line is named as well.
handles, labels = axis.get_legend_handles_labels()
salt_handles, salt_labels = salt_axis.get_legend_handles_labels()
axis.legend(handles + salt_handles, labels + salt_labels, loc="upper left")
axis.grid(True, color=sm.GRID_COLOR, lw=0.5)
figure.tight_layout()

## Key take away


- `IDA_ERR_FAIL` with `h` at its minimum says "even with the smallest time step, the error test which guarantees the accuracy of the time step failed"
- Solver failure with NAN or infinite values points to illegal operations such as taking the fractionated power of a negative value
- Solver failure at a time section transition points to consistent initialization
- The error message pointed at the time integrator, and changing the numerical input could resolve the issue
- A time integrator failure does not necessarily indicate a problem with solver settings: the actual fault was in the model
parameters

**Now:** return to [**99.7g User Stories**](../99.7g_user_stories_lesson.ipynb).